In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
TRAIN_PATH = r"C:\Users\yashs\Downloads\e88186124ec611f1\dataset\train.csv"
TEST_PATH = r"C:\Users\yashs\Downloads\e88186124ec611f1\dataset\test.csv"
OUT_PATH = r"C:\Users\yashs\Downloads\e88186124ec611f1\dataset\treeexp.csv"
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [3]:
def parse_time(x):
    h, m = map(int, str(x).split(":"))
    return h * 60 + m
base32 = "0123456789bcdefghjkmnpqrstuvwxyz"
def decode_geohash(g):
    lat = [-90.0, 90.0]
    lon = [-180.0, 180.0]
    even = True
    for ch in str(g):
        cd = base32.index(ch)
        for mask in [16, 8, 4, 2, 1]:
            if even:
                mid = (lon[0] + lon[1]) / 2
                if cd & mask:
                    lon[0] = mid
                else:
                    lon[1] = mid
            else:
                mid = (lat[0] + lat[1]) / 2
                if cd & mask:
                    lat[0] = mid
                else:
                    lat[1] = mid
            even = not even
    return (
        (lat[0] + lat[1]) / 2,
        (lon[0] + lon[1]) / 2
    )
geo_cache = {}

In [4]:
def make_features(df):
    df = df.copy()
    df["timestamp"] = df["timestamp"].astype(str)
    df["time_min"] = df["timestamp"].map(parse_time)
    df["hour"] = df["time_min"] // 60
    df["minute"] = df["time_min"] % 60
    # cyclic encoding
    df["time_sin"] = np.sin(2 * np.pi * df["time_min"] / 1440)
    df["time_cos"] = np.cos(2 * np.pi * df["time_min"] / 1440)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    # geohash hierarchy
    df["geo3"] = df["geohash"].astype(str).str[:3]
    df["geo4"] = df["geohash"].astype(str).str[:4]
    df["geo5"] = df["geohash"].astype(str).str[:5]
    cat_cols = [
        "geohash",
        "timestamp",
        "RoadType",
        "LargeVehicles",
        "Landmarks",
        "Weather",
        "geo3",
        "geo4",
        "geo5"
    ]
    for col in cat_cols:
        df[col] = (
            df[col]
            .astype(object)
            .where(df[col].notna(), "Missing")
            .astype(str)
        )
    # smarter temperature fill later
    df["temp_missing"] = df["Temperature"].isna().astype(int)
    return df
train_fe = make_features(train)
test_fe = make_features(test)

In [5]:
# smarter temperature imputation
temp_map = (
    train_fe
    .groupby(["geohash", "day"])["Temperature"]
    .median()
)
global_temp = train_fe["Temperature"].median()
temp_train = (
    train_fe
    .set_index(["geohash", "day"])
    .index
    .map(temp_map)
)
train_fe["Temperature"] = temp_train
train_fe["Temperature"] = train_fe["Temperature"].fillna(global_temp)
temp_test = (
    test_fe
    .set_index(["geohash", "day"])
    .index
    .map(temp_map)
)
test_fe["Temperature"] = temp_test
test_fe["Temperature"] = test_fe["Temperature"].fillna(global_temp)

In [6]:
# decode geohash
all_geo = pd.concat([
    train_fe["geohash"],
    test_fe["geohash"]
]).unique()
for g in all_geo:
    if g not in geo_cache:
        geo_cache[g] = decode_geohash(g)
train_fe["lat"] = train_fe["geohash"].map(lambda x: geo_cache[x][0])
train_fe["lon"] = train_fe["geohash"].map(lambda x: geo_cache[x][1])
test_fe["lat"] = test_fe["geohash"].map(lambda x: geo_cache[x][0])
test_fe["lon"] = test_fe["geohash"].map(lambda x: geo_cache[x][1])

In [7]:
# Agglo + KMeans spatial clustering
# DBSCAN removed because selection results showed it was weakest/noisy.
# We cluster unique geohashes only, then map cluster labels back to train/test rows.
geo_points = pd.concat([
    train_fe[["geohash", "lat", "lon"]],
    test_fe[["geohash", "lat", "lon"]]
]).drop_duplicates("geohash").reset_index(drop=True)
scaler = StandardScaler()
geo_scaled = scaler.fit_transform(
    geo_points[["lat", "lon"]]
)
cluster_cols = []
# KMeans clusters
for k in [10, 25, 50]:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    col = f"kmeans_{k}"
    geo_points[col] = km.fit_predict(geo_scaled).astype(str)
    mp = geo_points.set_index("geohash")[col]
    train_fe[col] = train_fe["geohash"].map(mp).astype(str)
    test_fe[col] = test_fe["geohash"].map(mp).astype(str)
    cluster_cols.append(col)
    print(f"\n{col} created")
    print("unique:", geo_points[col].nunique())
# DBSCAN removed because local validation ranked it weakest
# Agglomerative clusters
for k in [10, 25, 50]:
    ag = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )
    col = f"agglo_{k}"
    geo_points[col] = ag.fit_predict(geo_scaled).astype(str)
    mp = geo_points.set_index("geohash")[col]
    train_fe[col] = train_fe["geohash"].map(mp).astype(str)
    test_fe[col] = test_fe["geohash"].map(mp).astype(str)
    cluster_cols.append(col)
    print(f"\n{col} created")
    print("unique:", geo_points[col].nunique())

C:\Users\yashs\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(
C:\Users\yashs\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(



kmeans_10 created
unique: 10

kmeans_25 created
unique: 25


C:\Users\yashs\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(



kmeans_50 created
unique: 50

agglo_10 created
unique: 10

agglo_25 created
unique: 25

agglo_50 created
unique: 50


In [8]:
# cluster quality summary
for col in cluster_cols:
    print("\n==============================")
    print(col, "summary")
    print("==============================")
    temp = train_fe.groupby(col).agg(
        rows=("geohash", "count"),
        unique_geohash=("geohash", "nunique"),
        mean_demand=("demand", "mean"),
        min_lat=("lat", "min"),
        max_lat=("lat", "max"),
        min_lon=("lon", "min"),
        max_lon=("lon", "max")
    ).reset_index()
    print("Top demand clusters:")
    print(temp.sort_values("mean_demand", ascending=False).head(5))
    print("Size spread:")
    print(temp["rows"].describe())
    print("Demand spread:")
    print(temp["mean_demand"].describe())


kmeans_10 summary
Top demand clusters:
  kmeans_10   rows  unique_geohash  mean_demand   min_lat   max_lat  \
4         4  10546             145     0.153986 -5.408020 -5.320129   
8         8  11223             129     0.128331 -5.353088 -5.281677   
5         5   5964             122     0.109875 -5.484924 -5.402527   
6         6   7489             121     0.098598 -5.369568 -5.298157   
1         1  11603             130     0.074995 -5.413513 -5.353088   

     min_lon    max_lon  
4  90.708618  90.840454  
8  90.587769  90.719604  
5  90.642700  90.763550  
6  90.829468  90.972290  
1  90.587769  90.719604  
Size spread:
count       10.000000
mean      7729.900000
std       2465.853625
min       5041.000000
25%       5973.500000
50%       6971.000000
75%       9781.750000
max      11603.000000
Name: rows, dtype: float64
Demand spread:
count    10.000000
mean      0.088378
std       0.033152
min       0.058321
25%       0.062097
50%       0.074242
75%       0.107056
max       0.1

In [9]:
# interaction features
pairs = [
    ("geohash", "timestamp"),
    ("geohash", "RoadType"),
    ("geohash", "Weather"),
    ("geo4", "timestamp"),
    ("RoadType", "timestamp")
]
for col in cluster_cols:
    pairs.append((col, "timestamp"))
for a, b in pairs:
    train_fe[a + "_" + b] = (
        train_fe[a].astype(str)
        + "_"
        + train_fe[b].astype(str)
    )
    test_fe[a + "_" + b] = (
        test_fe[a].astype(str)
        + "_"
        + test_fe[b].astype(str)
    )

In [10]:
# KFold target encoding
target_cols = [
    ["geohash"],
    ["geohash", "timestamp"],
    ["geohash", "RoadType"],
    ["geohash", "Weather"],
    ["geo4", "timestamp"],
    ["RoadType", "timestamp"]
]
for col in cluster_cols:
    target_cols.append([col, "timestamp"])
global_mean = train_fe["demand"].mean()
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
for g in target_cols:
    col_name = "_".join(g) + "_te"
    train_fe[col_name] = np.nan
    for tr_idx, val_idx in kf.split(train_fe):
        tr_fold = train_fe.iloc[tr_idx]
        val_fold = train_fe.iloc[val_idx]
        mp = tr_fold.groupby(g)["demand"].mean()
        train_fe.iloc[val_idx,
                      train_fe.columns.get_loc(col_name)] = (
            val_fold
            .set_index(g)
            .index
            .map(mp)
            .fillna(global_mean)
        )
    full_map = train_fe.groupby(g)["demand"].mean()
    test_fe[col_name] = (
        test_fe
        .set_index(g)
        .index
        .map(full_map)
        .fillna(global_mean)
    )

In [11]:
# day49 continuation features
early_limit = parse_time("2:0")
early = train_fe[
    train_fe["time_min"] <= early_limit
].copy()
last_early = (
    early
    .sort_values("time_min")
    .groupby(["day", "geohash"])["demand"]
    .last()
)
train_fe["day_geo_last"] = (
    train_fe
    .set_index(["day", "geohash"])
    .index
    .map(last_early)
    .fillna(global_mean)
)
test_fe["day_geo_last"] = (
    test_fe
    .set_index(["day", "geohash"])
    .index
    .map(last_early)
    .fillna(global_mean)
)

In [12]:
# label encoding
cat_cols = [
    "geohash",
    "timestamp",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",
    "geo3",
    "geo4",
    "geo5",
    "geohash_timestamp",
    "geohash_RoadType",
    "geohash_Weather",
    "geo4_timestamp",
    "RoadType_timestamp"
]
for col in cluster_cols:
    cat_cols.append(col)
    cat_cols.append(col + "_timestamp")
features = [
    col for col in train_fe.columns
    if col not in ["Index", "demand"]
]
for col in cat_cols:
    if col in features:
        le = LabelEncoder()
        all_values = pd.concat([
            train_fe[col],
            test_fe[col]
        ]).astype(str)
        le.fit(all_values)
        train_fe[col] = le.transform(
            train_fe[col].astype(str)
        )
        test_fe[col] = le.transform(
            test_fe[col].astype(str)
        )
cat_used = [
    col for col in cat_cols
    if col in features
]

In [13]:
# weight recent rows higher
weights = np.where(
    train_fe["day"] == 49,
    3.0,
    1.0
)
model = lgb.LGBMRegressor(
    n_estimators=2500,
    learning_rate=0.015,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=35,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.4,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model.fit(
    train_fe[features],
    train_fe["demand"],
    sample_weight=weights,
    categorical_feature=cat_used
)

LGBMRegressor(colsample_bytree=0.85, learning_rate=0.015, min_child_samples=35,
              n_estimators=2500, n_jobs=-1, num_leaves=63, random_state=42,
              reg_alpha=0.4, reg_lambda=2.0, subsample=0.85, verbose=-1)

In [14]:
pred = model.predict(
    test_fe[features]
)
pred = np.clip(pred, 0, 1)
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": pred
})
submission.to_csv(
    OUT_PATH,
    index=False
)
print("saved:", OUT_PATH)
print(submission.head())
print("Prediction sanity check")
print("min:", pred.min())
print("max:", pred.max())
print("mean:", pred.mean())
print("std:", pred.std())
print("zero-ish:", (pred < 0.01).mean())
print("high > 0.5:", (pred > 0.5).mean())
print("high > 0.8:", (pred > 0.8).mean())

saved: C:\Users\yashs\Downloads\e88186124ec611f1\dataset\treeexp.csv
   Index    demand
0      0  0.062713
1      1  0.038132
2      2  0.041548
3      3  0.053988
4      4  0.101674
Prediction sanity check
min: 0.0
max: 1.0
mean: 0.1263663203945979
std: 0.16407158214306994
zero-ish: 0.024175403322322753
high > 0.5: 0.037244482742113075
high > 0.8: 0.016611613768011874
